# RNN for Sentiment Analysis Project Using  IMDB Dataset

# Load data

In [24]:
import pandas as pd

In [25]:
df = pd.read_csv("IMDB Dataset.csv")

In [26]:
df.shape

(50000, 2)

In [27]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [28]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [29]:
df.drop_duplicates(inplace=True)

In [30]:
df.shape

(49582, 2)

# Text Pre-Processing

### 1. Converting to lowercase

In [31]:
df["review"] = df["review"].str.lower()

### 2. Removing the URLs

In [32]:
import re

# sample_text = "abc is the word, abc" # abc => xyz

# new_text = re.sub("abc", "xyz", sample_text)

In [33]:
def remove_urls(text):
    text = re.sub(r"http\S+" , "", text) # (pattern, repl, string) eg - https://www.goole.com
    return text

df["review"] = df["review"].apply(remove_urls)

### 3. Removing Punctuations

In [34]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

df["review"] = df["review"].apply(remove_punctuations)

### 4. Removing HTML

In [35]:
def remove_html(text):
    text = re.sub(r"<.*?>" , "", text) 
    return text

df["review"] = df["review"].apply(remove_html)

### 5. Reomoving the Stopwords

In [36]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Raju\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Raju\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Raju\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [37]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [38]:
# sample_text = "I like coding in python!"
# tokens = word_tokenize(sample_text)

In [39]:
def remove_stopwords(text):
    todens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")

    return text

df["review"] = df["review"].apply(remove_stopwords)

In [40]:
df.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmg ...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love the time of money is a vi...,positive


### 6. Stemming

In [41]:
# running -> run
# played -> play
# PorterStemming

from nltk.stem import PorterStemmer

In [43]:
def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df["review"] = df["review"].apply(stemming)

In [44]:
df.head()

,review,sentiment
0,one of the other review ha mention that after ...,positive
1,a wonder littl product br br the filmg techniq...,positive
2,i thought thi wa a wonder way to spend time on...,positive
3,basic there a famili where a littl boy jake th...,negative
4,petter mattei love the time of money is a visu...,positive


### 7. Encoding

In [45]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df["sentiment"] = le.fit_transform(df["sentiment"])

In [46]:
y = df["sentiment"]

In [47]:
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### 8. Vectorization

In [48]:
df.head()

,review,sentiment
0,one of the other review ha mention that after ...,1
1,a wonder littl product br br the filmg techniq...,1
2,i thought thi wa a wonder way to spend time on...,1
3,basic there a famili where a littl boy jake th...,0
4,petter mattei love the time of money is a visu...,1


In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(df["review"])

# Train Test model

In [53]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [54]:
X_train.shape

(39665, 5000)

In [55]:
X_test.shape

(9917, 5000)

In [56]:
import torch
from torch.utils.data import TensorDataset, DataLoader

In [57]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [58]:
train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)

In [59]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

# Build our RNN

In [64]:
import torch.nn as nn
import torch.optim as optim

In [65]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()

        self.hidden_size = hidden_size
        self.num_layers = num_layers

        # RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # Fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0)
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

# Create a model

In [66]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

# Training the RNN

In [69]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction

        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")
        

epoch = 1/10 and loss = 0.29937678575515747
epoch = 2/10 and loss = 0.3103289306163788
epoch = 3/10 and loss = 0.23688086867332458
epoch = 4/10 and loss = 0.12500526010990143
epoch = 5/10 and loss = 0.2041955292224884
epoch = 6/10 and loss = 0.2640021741390228
epoch = 7/10 and loss = 0.23234768211841583
epoch = 8/10 and loss = 0.1404404193162918
epoch = 9/10 and loss = 0.15120632946491241
epoch = 10/10 and loss = 0.25656089186668396


# Evaluate

In [70]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0

    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

accuracy = 87.15337299586568
